In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import webbrowser

# ===== КОД КЛАССОВ =====

class Trader:
    def __init__(self, trader_id, capital, strategy, risk_tolerance):
        self.id = trader_id
        self.initial_capital = capital
        self.cash = capital
        self.strategy = strategy
        self.risk_tolerance = risk_tolerance
        self.position = 0

    @property
    def total_wealth(self):
        return self.cash + self.position * self.current_price

    def make_decision(self, current_price, fair_value, market_sentiment, price_history):
        self.current_price = current_price

        if self.strategy == 0:
            deviation = (current_price - fair_value) / fair_value
            if deviation < -0.05:
                return 1
            elif deviation > 0.05:
                return -1
        elif self.strategy == 1:
            if len(price_history) >= 3:
                momentum = (price_history[-1] - price_history[-3]) / price_history[-3]
                if momentum > 0.02 and market_sentiment > 0.6:
                    return 1
                elif momentum < -0.02 and market_sentiment < 0.4:
                    return -1
        elif self.strategy == 2:
            if market_sentiment > 0.7:
                return 1
            elif market_sentiment < 0.3:
                return -1
        return 0

    def update_strategy(self, other_traders):
        if np.random.random() < 0.05:
            other = np.random.choice(other_traders)
            if other.total_wealth > self.total_wealth:
                self.strategy = other.strategy

class Market:
    def __init__(self, num_traders=200, initial_price=100.0, fair_value=100.0):
        self.fair_value = fair_value
        self.price = initial_price
        self.price_history = [initial_price]

        strategies = np.random.choice([0, 1, 2], num_traders, p=[0.3, 0.4, 0.3])
        self.traders = [
            Trader(i, capital=10000, strategy=strategies[i], risk_tolerance=np.random.uniform(0.1, 1.0))
            for i in range(num_traders)
        ]

        self.sentiment_history = [0.5]
        self.volume_history = [0]
        self.strategy_history = [{0: 0.3, 1: 0.4, 2: 0.3}]

    def step(self):
        buy_orders = 0
        sell_orders = 0

        for trader in self.traders:
            sentiment = self.sentiment_history[-1]
            decision = trader.make_decision(
                self.price, self.fair_value, sentiment, self.price_history
            )

            if decision == 1 and trader.cash >= self.price:
                buy_orders += 1
            elif decision == -1 and trader.position > 0:
                sell_orders += 1

        total_volume = buy_orders + sell_orders + 1
        sentiment = buy_orders / total_volume

        noise = np.random.normal(0, 0.01)
        pressure = (buy_orders - sell_orders) / len(self.traders) * 0.3

        self.price = self.price * (1 + pressure + noise)
        self.price = max(self.price, 1.0)

        for trader in self.traders:
            decision = trader.make_decision(self.price, self.fair_value, sentiment, self.price_history)
            if decision == 1 and trader.cash >= self.price:
                trader.position += 1
                trader.cash -= self.price
            elif decision == -1 and trader.position > 0:
                trader.position -= 1
                trader.cash += self.price

        for trader in self.traders:
            trader.update_strategy(self.traders)

        self.price_history.append(self.price)
        self.sentiment_history.append(sentiment)
        self.volume_history.append(total_volume)

        strategies = [t.strategy for t in self.traders]
        self.strategy_history.append({
            0: strategies.count(0) / len(strategies),
            1: strategies.count(1) / len(strategies),
            2: strategies.count(2) / len(strategies)
        })

    def run_simulation(self, days=300):
        for _ in range(days):
            self.step()

# ===== ЗАПУСК СИМУЛЯЦИИ =====

print("Запуск симуляции...")
market = Market(num_traders=200, initial_price=100.0, fair_value=100.0)
market.run_simulation(days=300)

print(f"Симуляция завершена!")
print(f"Длина price_history: {len(market.price_history)}")
print(f"Длина sentiment_history: {len(market.sentiment_history)}")
print(f"Длина volume_history: {len(market.volume_history)}")
print(f"Длина strategy_history: {len(market.strategy_history)}")

Запуск симуляции...
Симуляция завершена!
Длина price_history: 301
Длина sentiment_history: 301
Длина volume_history: 301
Длина strategy_history: 301


In [4]:
# Создаем DataFrame для анализа
min_len = min(
    len(market.price_history),
    len(market.sentiment_history),
    len(market.volume_history),
    len(market.strategy_history)
)

df = pd.DataFrame({
    'day': range(min_len),
    'price': market.price_history[:min_len],
    'sentiment': market.sentiment_history[:min_len],
    'volume': market.volume_history[:min_len]
})

df['ma_20'] = df['price'].rolling(window=20).mean()
df['ma_50'] = df['price'].rolling(window=50).mean()
df['deviation'] = (df['price'] - market.fair_value) / market.fair_value * 100
df['bubble'] = df['price'] > (market.fair_value * 1.5)
df['crash'] = df['price'].pct_change(periods=10) < -0.3

print(f"Найдено пузырей: {df['bubble'].sum()} дней")
print(f"Найдено крахов: {df['crash'].sum()} дней")
df.head()

Найдено пузырей: 100 дней
Найдено крахов: 65 дней


,day,price,sentiment,volume,ma_20,ma_50,deviation,bubble,crash
0,0,100.000000,0.5,0,NaN,NaN,0.000000,False,False
1,1,101.036197,0.0,1,NaN,NaN,1.036197,False,False
2,2,101.475552,0.0,1,NaN,NaN,1.475552,False,False
3,3,101.013723,0.0,1,NaN,NaN,1.013723,False,False
4,4,100.561090,0.0,1,NaN,NaN,0.561090,False,False


In [5]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=('Цена акции с выделением пузырей',
                    'Сентимент рынка (доля покупателей)',
                    'Объем торгов'),
    row_heights=[0.6, 0.2, 0.2]
)

fig.add_trace(
    go.Scatter(x=df['day'], y=df['price'], mode='lines', name='Цена', line=dict(color='blue', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df['day'], y=[market.fair_value] * len(df), mode='lines', name='Fair Value', line=dict(color='red', dash='dash', width=2)),
    row=1, col=1
)

bubble_mask = df['bubble']
bubble_indices = df[bubble_mask]['day'].values

if len(bubble_indices) > 0:
    segments = []
    start = bubble_indices[0]
    prev = bubble_indices[0]

    for idx in bubble_indices[1:]:
        if idx - prev > 1:
            segments.append((start, prev))
            start = idx
        prev = idx
    segments.append((start, prev))

    for seg_start, seg_end in segments:
        fig.add_vrect(
            x0=seg_start, x1=seg_end,
            fillcolor="red", opacity=0.2,
            layer="below", line_width=0
        )

fig.add_trace(
    go.Scatter(x=df['day'], y=df['ma_20'], mode='lines', name='MA 20', line=dict(color='orange', width=1, dash='dot')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df['day'], y=df['sentiment'], mode='lines', name='Sentiment', fill='tozeroy', line=dict(color='purple', width=1)),
    row=2, col=1
)

fig.add_trace(
    go.Bar(x=df['day'], y=df['volume'], name='Volume', marker_color='gray'),
    row=3, col=1
)

fig.update_layout(
    title='Agent-Based Market Simulation - Bubble Detection',
    height=800, showlegend=True, template='plotly_white'
)

fig.update_xaxes(title_text="Days", row=3, col=1)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Sentiment", row=2, col=1)
fig.update_yaxes(title_text="Volume", row=3, col=1)

fig.show()

In [6]:
agent_data = []

for day in range(0, len(market.price_history), 5):
    price = market.price_history[day]

    for trader in market.traders:
        profit = trader.total_wealth - trader.initial_capital
        agent_data.append({
            'day': day,
            'risk_tolerance': trader.risk_tolerance,
            'profit': profit,
            'strategy': ['Rational', 'Emotional', 'Herd'][trader.strategy],
            'wealth': trader.total_wealth
        })

agent_df = pd.DataFrame(agent_data)

fig = px.scatter(
    agent_df,
    x='risk_tolerance',
    y='profit',
    color='strategy',
    animation_frame='day',
    title='Agent Movement: Risk vs Profit Over Time',
    labels={'risk_tolerance': 'Risk Tolerance', 'profit': 'Profit ($)', 'strategy': 'Strategy'},
    color_discrete_map={'Rational': 'blue', 'Emotional': 'red', 'Herd': 'green'}
)

fig.update_layout(height=600, template='plotly_white')
fig.show()

In [7]:
strategy_df = pd.DataFrame(market.strategy_history[:min_len])
strategy_df['day'] = range(len(strategy_df))

fig = go.Figure()

fig.add_trace(go.Scatter(x=strategy_df['day'], y=strategy_df[0], mode='lines', name='Rational', stackgroup='one', fillcolor='blue'))
fig.add_trace(go.Scatter(x=strategy_df['day'], y=strategy_df[1], mode='lines', name='Emotional', stackgroup='one', fillcolor='red'))
fig.add_trace(go.Scatter(x=strategy_df['day'], y=strategy_df[2], mode='lines', name='Herd', stackgroup='one', fillcolor='green'))

fig.update_layout(
    title='Strategy Distribution Over Time',
    yaxis_title='Proportion',
    xaxis_title='Days',
    template='plotly_white',
    height=500
)

fig.show()